# Custom Evaluators: Minimizing CPU Time and Memory in Chemistry

This notebook optimizes Trotter dynamics for the DMDS molecule (C₂H₆S₂) by:
- **Minimizing CPU time** - actual process CPU consumption (not wall-clock)
- **Minimizing memory usage** - resident set size (RSS)
- **Maintaining accuracy**: error with exact dynamics < 0.1

Uses `trv.eval.CPUTime` and `trv.eval.Memory` constraints that monitor real system resources.

In [1]:
import numpy as np
import scipy
import scipy.sparse as sp
import trovolut as trv
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

## 1. Create DMDS Hamiltonian

Dimethyl disulfide (DMDS, C₂H₆S₂) - a real chemistry problem.

In [2]:
from rdkit import Chem
from rdkit.Chem import AllChem

# Generate DMDS geometry
dmds_smiles = "CSSC"
mol = Chem.MolFromSmiles(dmds_smiles)
mol = Chem.AddHs(mol)

AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
AllChem.UFFOptimizeMolecule(mol)

# Extract XYZ
atoms = [atom.GetSymbol() for atom in mol.GetAtoms()]
conf = mol.GetConformer()

xyz_lines = [str(len(atoms)), "dmds generated by RDKit"]
for i, sym in enumerate(atoms):
    pos = conf.GetAtomPosition(i)
    xyz_lines.append(f"{sym} {pos.x:.8f} {pos.y:.8f} {pos.z:.8f}")

dmds_xyz = "\n".join(xyz_lines)
print(f"Generated {len(atoms)} atom DMDS molecule")

Generated 10 atom DMDS molecule


In [3]:
# Create electronic Hamiltonian
H_elec = trv.qm.ElectronicHamiltonian.from_xyz(
    xyz_source=dmds_xyz,
    basis="sto-3g",
    charge=0,
    spin_mult=1,
    use_gpu4pyscf=False,
    freeze_core=True,
    n_core_keep=1,  # Just HOMO
    n_virt_keep=1,  # Just LUMO -> 2 spatial orbs = 4 qubits (dim 16)
)

# Convert to qubit Hamiltonian (sparse for efficiency)
H_qubit = H_elec.to_qubits(mapping="jordan-wigner", sparse=True)

qmeta = H_qubit.decompose()
print(f"Qubits: {qmeta['n_qubits']}")
print(f"Pauli terms: {len(qmeta['terms'])}")
print(f"Hilbert space dimension: 2^{qmeta['n_qubits']} = {2**qmeta['n_qubits']:,}")

converged SCF energy = -864.654938672562
Qubits: 8
Pauli terms: 31
Hilbert space dimension: 2^8 = 256


In [4]:
def get_hamiltonian():
    """Return Hamiltonian terms (cached, sparse matrices)."""
    return [t["coef"] * t["matrix"] for t in H_qubit.terms()]

hams = get_hamiltonian()
n_qubits = qmeta['n_qubits']
dim = 2**n_qubits

print(f"\nHamiltonian properties:")
print(f"  Terms: {len(hams)}")
print(f"  Matrix dimension: {dim} × {dim}")
print(f"  Matrix type: {type(hams[0])}")
if sp.issparse(hams[0]):
    print(f"  Sparsity: {hams[0].nnz / (dim * dim) * 100:.2f}% non-zero")


Hamiltonian properties:
  Terms: 31
  Matrix dimension: 256 × 256
  Matrix type: <class 'scipy.sparse._csr.csr_matrix'>
  Sparsity: 0.39% non-zero


## 2. Configure Resource Constraints

We use two built-in resource monitoring constraints:
- **`CPUTime`**: Monitors actual CPU time (user + system) via `psutil`
- **`Memory`**: Tracks memory usage (RSS) during execution

Both are robust - they measure real system resources and cannot be bypassed.

## 3. Configure Error Measurement

In [7]:
def setup():
    """Return inputs for dynamics: (hams, t, psi0)."""
    hams = get_hamiltonian()
    
    # Random initial state
    psi0 = np.random.randn(dim) + 1j * np.random.randn(dim)
    psi0 /= np.linalg.norm(psi0)
    
    return hams, 1.0, psi0

def exact_evolution(hams, t, psi0):
    """Exact time evolution for reference."""
    H_total = sum(hams)
    if sp.issparse(H_total):
        # Use sparse exponential action
        return sp.linalg.expm_multiply(-1j * t * H_total, psi0)
    else:
        return scipy.linalg.expm(-1j * t * H_total) @ psi0

print("✓ Setup and exact evolution functions defined")


✓ Error measurement configured


In [ ]:
# Create constraint set with built-in resource monitoring
constraints = trv.eval.ConstraintSet([
    trv.eval.CPUTime(max_cpu_ms=100, weight=2.0, score_fn='exponential'),
    trv.eval.Memory(max_mb=100, weight=1.0, score_fn='linear'),
    trv.eval.Error(max_error=0.1, hard=True, hard_penalty=-100.0)
])

# Configure error measurement
constraints.config_error(
    exact_fn=exact_evolution,
    setup_fn=setup
).entry_point('run_dynamics')

print("\n" + "="*70)
print("CONSTRAINT CONFIGURATION")
print("="*70)
for i, c in enumerate(constraints.constraints, 1):
    print(f"\n{i}. {c.__class__.__name__} ('{c.name}')")
    print(f"   Target: {getattr(c, 'max_cpu_ms', None) or getattr(c, 'max_mb', None) or getattr(c, 'max_error', None)}")
    print(f"   Weight: {c.weight}")
    print(f"   Hard: {c.hard}")
    print(f"   Prompt: {c.to_prompt()}")
    print(f"   Formula: {c.formula()}")
print("\n" + "="*70)


## 4. Generate Files with Workflow

Initial program uses standard Trotter with fp64 dense matrices.

In [ ]:
Path("chem_opt").mkdir(exist_ok=True)

def run_dynamics(hams, t, psi0):
    """
    Standard first-order Trotter evolution.
    Initial implementation uses dense fp64 matrices - optimizer should improve this!
    """
    n_steps = 50
    dt = t / n_steps
    psi = psi0.copy()
    
    for _ in range(n_steps):
        for H in hams:
            # Convert to dense if sparse (naive initial approach)
            if sp.issparse(H):
                H_dense = H.toarray()
            else:
                H_dense = H
            
            # Compute exponential and apply
            U = scipy.linalg.expm(-1j * H_dense * dt)
            psi = U @ psi
    
    return psi

# Build workflow
workflow = trv.filemaker.Workflow(
    [
    trv.filemaker.EVO_START, 
    run_dynamics, 
    trv.filemaker.EVO_END
    ],
    out_path="chem_opt/initial_program.py",
    imports=[
        "import scipy",
        "import scipy.linalg",
        "import scipy.sparse as sp",
        "import scipy.sparse.linalg",
        "import numpy as np",
    ],
    constraints=constraints
)

# Generate files
init_path = workflow.build()
evaluator_path = workflow.build_evaluator()

print(f"Generated: {init_path}")
print(f"Generated: {evaluator_path}")


Initial:   chem_opt/initial_program.py
Evaluator: chem_opt/evaluator.py


## 5. Test Evaluator

In [9]:
# Load and test
exec(open(evaluator_path).read())

result = evaluate(str(init_path))

print("="*60)
print(" INITIAL PROGRAM EVALUATION")
print("="*60)
print(f"Combined score: {result['combined_score']:.4f}")
print(f"All satisfied: {result['all_satisfied']}")
print("\nConstraint details:")
for name, info in result['details'].items():
    print(f"  {name}: value={info['value']:.4f}, score={info['score']:.4f}")

print("\n" + "="*60)
print(" OPTIMIZATION OPPORTUNITIES")
print("="*60)
print("The initial program uses:")
print("  • Dense matrices (convert from sparse)")
print("  • fp64 precision (complex128)")
print("  • Full matrix exponentials")
print("\nPotential improvements:")
print("  ✓ Keep matrices sparse (avoid .toarray())")
print("  ✓ Use sp.linalg.expm_multiply() for sparse matrices")
print("  ✓ Reduce precision (complex128 → complex64)")
print("  ✓ Use approximations (fewer Trotter steps, Taylor series)")
print("  ✓ Cache exponentials if same dt used")
print("="*60)

NameError: name 'flop_counter' is not defined

## 6. Run OpenEvolve

Let evolution discover methods to reduce FLOPs while maintaining accuracy.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, "-m", "openevolve",
    "--init", str(init_path),
    "--eval", str(evaluator_path),
    "--outdir", "chem_opt/evolution_results",
    "--model", "gpt-4o-mini",
    "--iters", "100",
    "--popsize", "10",
    "--feature-dim", "2",
    "--save-every", "10",
]

print("Starting OpenEvolve...")
print(" ".join(cmd))
subprocess.run(cmd, check=True)

## 7. Analyze Results

In [ ]:
import json
import matplotlib.pyplot as plt

results_dir = Path("chem_opt/evolution_results")
checkpoints_dir = results_dir / "checkpoints"

# Load checkpoint data
checkpoints = []
for ckpt_dir in sorted(checkpoints_dir.glob("checkpoint_*"), key=lambda x: int(x.name.split("_")[1])):
    metadata_file = ckpt_dir / "metadata.json"
    if metadata_file.exists():
        with open(metadata_file) as f:
            data = json.load(f)
            checkpoints.append(data)

print(f"Loaded {len(checkpoints)} checkpoints")

# Extract metrics
iterations = [c['iteration'] for c in checkpoints]
best_scores = [c['best_score'] for c in checkpoints]
best_flops = [c['best_details']['flops']['value'] for c in checkpoints]
best_errors = [c['best_details']['error']['value'] for c in checkpoints]

In [ ]:
# Plot evolution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Combined score
axes[0, 0].plot(iterations, best_scores, 'o-', linewidth=2, markersize=6)
axes[0, 0].set_xlabel('Iteration', fontsize=12)
axes[0, 0].set_ylabel('Combined Score', fontsize=12)
axes[0, 0].set_title('Evolution: Combined Score', fontsize=14, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# FLOPs (log scale)
axes[0, 1].plot(iterations, best_flops, 'o-', color='orange', linewidth=2, markersize=6)
axes[0, 1].set_xlabel('Iteration', fontsize=12)
axes[0, 1].set_ylabel('GFLOPs', fontsize=12)
axes[0, 1].set_yscale('log')
axes[0, 1].set_title('Evolution: Floating Point Operations (log scale)', fontsize=14, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, which='both')

# Error
axes[1, 0].plot(iterations, best_errors, 'o-', color='green', linewidth=2, markersize=6)
axes[1, 0].axhline(y=0.1, color='r', linestyle='--', linewidth=2, label='Max error (0.1)')
axes[1, 0].set_xlabel('Iteration', fontsize=12)
axes[1, 0].set_ylabel('Error', fontsize=12)
axes[1, 0].set_yscale('log')
axes[1, 0].set_title('Evolution: Simulation Error (log scale)', fontsize=14, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3, which='both')

# Pareto front
scatter = axes[1, 1].scatter(best_flops, best_errors, c=iterations, 
                             cmap='viridis', s=100, alpha=0.7, edgecolors='black')
axes[1, 1].axhline(y=0.1, color='r', linestyle='--', alpha=0.5, linewidth=2, label='Max error')
axes[1, 1].set_xlabel('GFLOPs', fontsize=12)
axes[1, 1].set_ylabel('Error', fontsize=12)
axes[1, 1].set_xscale('log')
axes[1, 1].set_yscale('log')
axes[1, 1].set_title('Pareto Front: FLOPs vs Error', fontsize=14, fontweight='bold')
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(True, alpha=0.3, which='both')
cbar = plt.colorbar(scatter, ax=axes[1, 1])
cbar.set_label('Iteration', fontsize=11)

plt.tight_layout()
plt.savefig('chem_opt/evolution_results/evolution_plots.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print(" OPTIMIZATION RESULTS")
print("="*60)
print(f"Initial → Final:")
print(f"  Score:  {best_scores[0]:.4f} → {best_scores[-1]:.4f} ({(best_scores[-1]/best_scores[0]-1)*100:+.1f}%)")
print(f"  FLOPs:  {best_flops[0]:.2f} → {best_flops[-1]:.2f} GFLOPs ({(1-best_flops[-1]/best_flops[0])*100:.1f}% reduction)")
print(f"  Error:  {best_errors[0]:.4e} → {best_errors[-1]:.4e}")
print("="*60)

## 8. Generate Code Evolution GIF

In [ ]:
from trovolut.utils import create_diff_gif

create_diff_gif(
    run_dir=Path("chem_opt"),
    width=1400,
    line_height=30,
    font_size=16,
    theme='dark',
    include_final_best=True
)

## 9. Inspect Best Solution

In [ ]:
# Read the final best program
best_program_path = results_dir / "best" / "best_program.py"
if best_program_path.exists():
    print("="*60)
    print(" FINAL BEST PROGRAM")
    print("="*60)
    with open(best_program_path, 'r') as f:
        print(f.read())
    print("="*60)
    
    # Load and check what optimizations were discovered
    with open(results_dir / "best" / "best_program_info.json", 'r') as f:
        best_info = json.load(f)
    
    print("\nBest program details:")
    print(f"  Score: {best_info['score']:.4f}")
    print(f"  FLOPs: {best_info['details']['flops']['value']:.2f} GFLOPs")
    print(f"  Error: {best_info['details']['error']['value']:.4e}")
    print(f"  Satisfies constraints: {best_info['all_satisfied']}")